# Accelerating PyTorch model training on Huawei Ascend NPUs

[PyTorch](https://pytorch.org/) is an open source deep learning framework in Python originally developed by [Meta](https://www.meta.com/), now hosted under the vendor-neutral [PyTorch Foundation](https://pytorch.org/foundation/). It is the de-facto industry standard for training modern deep learning models and large language models \(LLMs\). Like most other deep learning frameworks, PyTorch natively supports accelerating machine learning workloads on NVIDIA GPUs through its [CUDA](https://developer.nvidia.com/cuda/toolkit) ecosystem without the need for special plugins or adapters.

The [`torch-npu`](https://pypi.org/project/torch-npu/) plugin developed by the Huawei [Ascend](https://www.hiascend.com/en) community enables AI/ML engineers to migrate machine learning workloads from NVIDIA GPUs to Ascend NPUs seamlessly with minimal changes to existing code and processes. With the high-level model training logic in PyTorch unchanged, the [CANN](https://www.hiascend.com/en/cann) kernels library is used in place of CUDA and Ascend NPUs are used for hardware acceleration in place of NVIDIA GPUs.

This notebook experiment serves as a gentle introduction to training deep learning models with PyTorch on Ascend NPUs, using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset as our motivating example.

## Python version and dependencies

This notebook experiment runs on Python 3.11 with the following Python package dependencies.

1. PyTorch 2.8.0
1. [PyYAML](https://pypi.org/project/PyYAML/) 6.0.3
1. [setuptools](https://pypi.org/project/setuptools/) 82.0.1
1. `torch-npu` 2.8.0
1. CANN 8.5.0

In [1]:
!cat requirements.txt

absl-py==2.4.0
attrs==25.4.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.6
jupyterlab-git==0.52.0
jupyter-resource-usage==1.2.0
ml-dtypes==0.5.4
torch==2.8.0
torch-npu==2.8.0
pyyaml==6.0.3
scipy==1.17.1
setuptools==82.0.1
sympy==1.14.0
tornado==6.5.5


In [2]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Confirming that NPU acceleration is available

The `torch.npu.is_available` method reports whether Ascend NPU acceleration is available.

In [3]:
import torch
import torch_npu

torch.npu.is_available()

/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0/aarch64-linux/ascend_toolkit_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/ora

True

Warnings can be safely ignored unless you see messages starting with `[ERROR]` or `[CRITICAL]` in which case consult the Ascend forum for assistance.

As shown in the output above, NPU acceleration is available on our [OrangePi AIpro \(20T\)](http://www.orangepi.org/html/hardWare/computerAndMicrocontrollers/details/Orange-Pi-AIpro%2820t%29.html) development board which includes a single Ascend 310B1 NPU chip and core.

## Moving tensors to NPU

TODO